Simple Circuit File Reader - Read QASM files into QuantumCircuit objects

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(f"./")

In [2]:
import os
import glob
from qiskit import QuantumCircuit, qasm2

In [3]:
def read_circuit(filepath: str) -> QuantumCircuit:
    """
    Read a quantum circuit from a QASM file.
    
    Args:
        filepath: Path to the .qasm file
        
    Returns:
        QuantumCircuit object
    """
    return qasm2.load(filepath)


def find_debug_files(debug_dir: str = "mcrx_optimization_errors"):
    """Find all available debug files in the directory."""
    if not os.path.exists(debug_dir):
        print(f"Directory '{debug_dir}' doesn't exist.")
        return []
    
    # Find all .qasm files
    qasm_files = glob.glob(os.path.join(debug_dir, "*.qasm"))
    return qasm_files


def read_latest_circuit(debug_dir: str = "mcrx_optimization_errors"):
    """Read the most recent circuit file."""
    files = find_debug_files(debug_dir)
    if not files:
        print("No .qasm files found.")
        return None
    
    # Sort by modification time (most recent first)
    files.sort(key=os.path.getmtime, reverse=True)
    latest_file = files[0]
    
    print(f"Reading: {latest_file}")
    return read_circuit(latest_file)


def print_circuit_gates(circuit: QuantumCircuit) -> None:
    """
    Print circuit as a sequence of gates instead of a diagram.
    
    Args:
        circuit: QuantumCircuit to analyze
    """
    print(f"Circuit with {len(circuit.data)} gates:")
    print("=" * 50)
    
    for i, instruction in enumerate(circuit.data):
        gate_name = instruction.operation.name
        qubits = [circuit.find_bit(q).index for q in instruction.qubits]
        
        # Get parameters if available
        params = []
        if hasattr(instruction.operation, 'params') and instruction.operation.params:
            for param in instruction.operation.params:
                if hasattr(param, 'evalf'):  # SymPy expression
                    params.append(float(param.evalf()))
                else:
                    params.append(float(param))
        
        # Format output
        param_str = ""
        if params:
            param_str = f"({', '.join(f'{p:.4f}' for p in params)})"
        
        qubit_str = f"qubits: {qubits}"
        
        print(f"Gate {i+1}: {gate_name}{param_str} -> {qubit_str}")
        
        # Additional info for MCRX gates
        if 'rx' in gate_name.lower() and len(qubits) > 1:
            control_qubits = qubits[:-1]
            target_qubit = qubits[-1]
            print(f"         Controls: {control_qubits} -> Target: {target_qubit}")
            
            # Try to extract control pattern
            if hasattr(instruction.operation, 'ctrl_state'):
                ctrl_state = instruction.operation.ctrl_state
                print(f"         Control pattern: {ctrl_state}")


def print_circuit_sequence(circuit: QuantumCircuit) -> None:
    """
    Print circuit as a simple gate sequence.
    
    Args:
        circuit: QuantumCircuit to display
    """
    print("Gate Sequence:")
    print("-" * 30)
    
    for i, instruction in enumerate(circuit.data):
        gate_name = instruction.operation.name
        qubits = [circuit.find_bit(q).index for q in instruction.qubits]
        
        # Get angle parameter
        angle_str = ""
        if hasattr(instruction.operation, 'params') and instruction.operation.params:
            angle = instruction.operation.params[0]
            if hasattr(angle, 'evalf'):
                angle_val = float(angle.evalf())
            else:
                angle_val = float(angle)
            angle_str = f"(θ={angle_val:.4f})"
        
        print(f"{i+1}. {gate_name}{angle_str} {qubits}")


def analyze_mcrx_circuit(circuit: QuantumCircuit) -> None:
    """
    Detailed analysis of MCRX circuit showing patterns and structure.
    
    Args:
        circuit: QuantumCircuit to analyze
    """
    print("MCRX Circuit Analysis:")
    print("=" * 40)
    print(f"Total gates: {len(circuit.data)}")
    print(f"Total qubits: {circuit.num_qubits}")
    print(f"Circuit depth: {circuit.depth()}")
    print()
    
    # Analyze each gate
    mcrx_gates = []
    other_gates = []
    
    for i, instruction in enumerate(circuit.data):
        gate_info = {
            'index': i,
            'name': instruction.operation.name,
            'qubits': [circuit.find_bit(q).index for q in instruction.qubits],
            'params': []
        }
        
        # Extract parameters
        if hasattr(instruction.operation, 'params') and instruction.operation.params:
            for param in instruction.operation.params:
                if hasattr(param, 'evalf'):
                    gate_info['params'].append(float(param.evalf()))
                else:
                    gate_info['params'].append(float(param))
        
        # Extract control state if available
        if hasattr(instruction.operation, 'ctrl_state'):
            gate_info['ctrl_state'] = instruction.operation.ctrl_state
        
        # Categorize gates
        if 'rx' in instruction.operation.name.lower():
            mcrx_gates.append(gate_info)
        else:
            other_gates.append(gate_info)
    
    # Print MCRX gates
    if mcrx_gates:
        print("MCRX Gates:")
        for gate in mcrx_gates:
            angle_str = f"θ={gate['params'][0]:.4f}" if gate['params'] else "no angle"
            ctrl_qubits = gate['qubits'][:-1] if len(gate['qubits']) > 1 else []
            target_qubit = gate['qubits'][-1] if gate['qubits'] else None
            
            print(f"  {gate['index']+1}. {gate['name']} ({angle_str})")
            print(f"      Controls: {ctrl_qubits} -> Target: {target_qubit}")
            
            if 'ctrl_state' in gate:
                print(f"      Pattern: {gate['ctrl_state']}")
            print()
    
    # Print other gates
    if other_gates:
        print("Other Gates:")
        for gate in other_gates:
            param_str = ""
            if gate['params']:
                param_str = f"({', '.join(f'{p:.4f}' for p in gate['params'])})"
            
            print(f"  {gate['index']+1}. {gate['name']}{param_str} -> {gate['qubits']}")

In [5]:
# List available files
print("Available debug files:")
files = find_debug_files()
for i, file in enumerate(files):
    print(f"  {i}: {file}")

if files:
    # Read a circuit file
    circuit = read_circuit(files[1])  # Change index as needed
    
    print(f"\n" + "="*60)
    print("CIRCUIT ANALYSIS")
    print("="*60)
    
    # Method 1: Simple gate sequence
    print_circuit_sequence(circuit)
    print()
    
    # Method 2: Detailed gate information
    print_circuit_gates(circuit)
    print()
    
    # Method 3: MCRX-specific analysis
    analyze_mcrx_circuit(circuit)
    
    # Method 4: Text diagram (alternative to matplotlib)
    print("\nText Diagram:")
    print(circuit.draw())
    
else:
    print("\nNo files found. Run MCRX optimization to generate debug files.")


Available debug files:
  0: mcrx_optimization_errors\optimized_circuit_20250609_164704.qasm
  1: mcrx_optimization_errors\original_circuit_20250609_164704.qasm

CIRCUIT ANALYSIS
Gate Sequence:
------------------------------
1. c8rx_o253(θ=0.7854) [0, 1, 2, 3, 4, 5, 6, 7, 8]
2. c8rx_o87(θ=0.7854) [0, 1, 2, 3, 4, 5, 6, 7, 8]

Circuit with 2 gates:
Gate 1: c8rx_o253(0.7854) -> qubits: [0, 1, 2, 3, 4, 5, 6, 7, 8]
         Controls: [0, 1, 2, 3, 4, 5, 6, 7] -> Target: 8
Gate 2: c8rx_o87(0.7854) -> qubits: [0, 1, 2, 3, 4, 5, 6, 7, 8]
         Controls: [0, 1, 2, 3, 4, 5, 6, 7] -> Target: 8

MCRX Circuit Analysis:
Total gates: 2
Total qubits: 9
Circuit depth: 2

MCRX Gates:
  1. c8rx_o253 (θ=0.7854)
      Controls: [0, 1, 2, 3, 4, 5, 6, 7] -> Target: 8

  2. c8rx_o87 (θ=0.7854)
      Controls: [0, 1, 2, 3, 4, 5, 6, 7] -> Target: 8


Text Diagram:
     ┌─────────────────┐┌────────────────┐
q_0: ┤0                ├┤0               ├
     │                 ││                │
q_1: ┤1            